# 03 — Beijing EDA

EDA từ working clean dataset; figure/table chỉ lưu trong son_nb/output.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ------------------------------------------------------------
# Robust project root finder
# Notebook lives under: Project/notebooks/son_nb/
# ------------------------------------------------------------
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "raw").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        "Không tìm thấy Project root. Hãy chạy notebook bên trong project."
    )

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAIN_OUTPUT = PROJECT_ROOT / "outputs"
SON_DIR = PROJECT_ROOT / "notebooks" / "son_nb"
SON_OUTPUT = SON_DIR / "output"

WORK_DATA = SON_OUTPUT / "data"
WORK_FIG = SON_OUTPUT / "figures"
WORK_TABLE = SON_OUTPUT / "tables"
WORK_PRED = SON_OUTPUT / "predictions"
WORK_MODEL = SON_OUTPUT / "models"

for p in [WORK_DATA, WORK_FIG, WORK_TABLE, WORK_PRED, WORK_MODEL]:
    p.mkdir(parents=True, exist_ok=True)

BEIJING_RAW = RAW_DIR / "beijing" / "PRSA_data_2010.1.1-2014.12.31.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BEIJING_RAW:", BEIJING_RAW)
print("Working output:", SON_OUTPUT)

In [ ]:
df=pd.read_csv(WORK_DATA/'beijing_hourly_clean_working.csv',parse_dates=['datetime'])

In [ ]:
daily=df.set_index('datetime')['pm2.5'].resample('D').mean()
fig,ax=plt.subplots(figsize=(14,5)); ax.plot(daily.index,daily.values); ax.set_title('Beijing — Daily Mean PM2.5'); ax.set_xlabel('Date'); ax.set_ylabel('PM2.5'); fig.tight_layout(); fig.savefig(WORK_FIG/'01_beijing_daily_pm25.png',dpi=200); plt.show()

In [ ]:
monthly=df.assign(month_period=df.datetime.dt.to_period('M')).groupby('month_period')['pm2.5'].mean()
fig,ax=plt.subplots(figsize=(14,5)); ax.plot(monthly.index.astype(str),monthly.values); ax.set_title('Beijing — Monthly Mean PM2.5'); ax.tick_params(axis='x',rotation=90); ax.set_ylabel('PM2.5'); fig.tight_layout(); fig.savefig(WORK_FIG/'02_beijing_monthly_pm25.png',dpi=200); plt.show()

In [ ]:
hour=df.groupby('hour')['pm2.5'].mean()
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(hour.index,hour.values,marker='o'); ax.set_title('Beijing — Mean PM2.5 by Hour'); ax.set_xlabel('Hour'); ax.set_ylabel('PM2.5'); fig.tight_layout(); fig.savefig(WORK_FIG/'03_beijing_pm25_by_hour.png',dpi=200); plt.show()

In [ ]:
cols=['pm2.5','DEWP','TEMP','PRES','Iws','Is','Ir']; corr=df[cols].corr(); fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(corr.values,aspect='auto'); ax.set_xticks(range(len(cols))); ax.set_yticks(range(len(cols))); ax.set_xticklabels(cols,rotation=45,ha='right'); ax.set_yticklabels(cols); ax.set_title('Correlation Matrix'); fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(WORK_FIG/'04_beijing_correlation_matrix.png',dpi=200); plt.show(); corr.to_csv(WORK_TABLE/'beijing_correlation_matrix.csv')

In [ ]:
summary=df[cols].describe().T; display(summary); summary.to_csv(WORK_TABLE/'beijing_eda_summary.csv')

### Gate 03
EDA cho Chương 4.1. Không suy diễn nhân quả từ correlation.